In [4]:
import any_db_connector

import logging
import datetime
from typing import List, Dict, Tuple, Optional, Any
from concurrent.futures import ThreadPoolExecutor, as_completed


In [46]:
any_db_con =DBConFactory()

In [61]:
## 프로파일링 실행 소스DB
any_db_con.register_oracle(
    conn_id = "Oracle_DB1"
    , host = "172.16.105.117"
    , port = "1521"
    , user = "E2218030"
    , password = "password"
    , service_name = "PDB1"
    , sid = "PDB1"
)

## 프로파일링 결과 저장DB
any_db_con.register_postgres(
    conn_id= "AR_DB"
    , host = "172.16.105.117"
    , port= "5432"
    , database = "postgres"
    , user = "e2218030"
    , password = "p2218030"
)

any_db_con.list()

2026-04-29 08:49:41,039 [WARNING] [Oracle_DB1] 이미 등록된 conn_id — 기존 반환
2026-04-29 08:49:41,041 [WARNING] [AR_DB] 이미 등록된 conn_id — 기존 반환


['Oracle_DB1', 'AR_DB']

In [96]:
target_db = any_db_con.get("Oracle_DB1")
AR_DB = any_db_con.get("AR_DB")

In [89]:
AR_DB.execute_dml(query="select 1 ")

1

In [ ]:
ORACLE_EXCL = "'SYS','SYSCAT','SYSGIS','SYSTEM','OUTLN','DBSNMP','APPQOSSYS','WMSYS','XDB'"
PG_EXCL     = "'pg_catalog','information_schema','pg_toast'"
MYSQL_EXCL  = "'information_schema','performance_schema','mysql','sys'"
MSSQL_EXCL  = "'master','tempdb','model','msdb'

## TEST

In [141]:
TARGET_ORACLE_SCHEMA

['E2218030', 'E2618005']

In [ ]:
from any_db_connector import (BaseConnector, OracleConnector, PostgresConnector,
                               MariaDBConnector, MSSQLConnector, DBConFactory)


ORACLE_EXCL = "'SYS','SYSCAT','SYSGIS','SYSTEM','OUTLN','DBSNMP','APPQOSSYS','WMSYS','XDB'"
PG_EXCL     = "'pg_catalog','information_schema','pg_toast'"
MYSQL_EXCL  = "'information_schema','performance_schema','mysql','sys'"
MSSQL_EXCL  = "'master','tempdb','model','msdb'"

TARGET_ORACLE_SCHEMA =['E2218030','E2618005']

def _flavor(conn: BaseConnector) -> str:
    """커넥터 종류 반환 : 'oracle' | 'postgres' | 'mysql' | 'mssql'"""
    if isinstance(conn, OracleConnector):   return "oracle"
    if isinstance(conn, PostgresConnector): return "postgres"
    if isinstance(conn, MariaDBConnector):  return "mysql"
    if isinstance(conn, MSSQLConnector):    return "mssql"
    return "oracle"


In [ ]:
oradb = _flavor(target_db)
ardb = _flavor(AR_DB)

In [132]:
diccol = DicCollector(target_db, AR_DB, db_nm ='oracle_db') 

In [142]:
diccol._reg_chasu(chasu=1)

DatabaseError: DPY-4011: the database or network closed the connection
[WinError 10054] 현재 연결은 원격 호스트에 의해 강제로 끊겼습니다

Exception ignored in: 'oracledb.thin_impl.Protocol._send_marker'
Traceback (most recent call last):
  File "src\oracledb\impl/thin/packet.pyx", line 644, in oracledb.thin_impl.WriteBuffer.end_request
  File "src\oracledb\impl/thin/packet.pyx", line 623, in oracledb.thin_impl.WriteBuffer._send_packet
  File "c:\Users\ISKIM\anaconda3\Lib\site-packages\oracledb\errors.py", line 118, in _raise_err
    raise exc_type(_Error(message)) from cause
oracledb.exceptions.DatabaseError: DPY-4011: the database or network closed the connection
[WinError 10054] 현재 연결은 원격 호스트에 의해 강제로 끊겼습니다


DatabaseError: DPY-4011: the database or network closed the connection
[WinError 10054] 현재 연결은 원격 호스트에 의해 강제로 끊겼습니다

Exception ignored in: 'oracledb.thin_impl.Protocol._send_marker'
Traceback (most recent call last):
  File "src\oracledb\impl/thin/packet.pyx", line 644, in oracledb.thin_impl.WriteBuffer.end_request
  File "src\oracledb\impl/thin/packet.pyx", line 623, in oracledb.thin_impl.WriteBuffer._send_packet
  File "c:\Users\ISKIM\anaconda3\Lib\site-packages\oracledb\errors.py", line 118, in _raise_err
    raise exc_type(_Error(message)) from cause
oracledb.exceptions.DatabaseError: DPY-4011: the database or network closed the connection
[WinError 10054] 현재 연결은 원격 호스트에 의해 강제로 끊겼습니다
2026-04-29 13:47:45,935 [ERROR] [Oracle_DB1] 롤백: DPY-4009: 0 positional bind values are required but 2 were provided


DatabaseError: DPY-4009: 0 positional bind values are required but 2 were provided

In [ ]:
# ══════════════════════════════════════════════════════════════
# Step 1 : 딕셔너리 수집
# ══════════════════════════════════════════════════════════════
class DicCollector:
    """
    target DB(Oracle/PostgreSQL)에서 딕셔너리를 수집해
    result DB(PostgreSQL) ENC_DIC_* 테이블에 저장한다.

    사용 예시:
        dc = DicCollector(target=oracle_conn, result=pg_conn, db_nm="01", schema="public")
        chasu = dc.collect()
    """

    def __init__(self, target: BaseConnector, result: BaseConnector, db_nm: str):
        self.target = target
        self.result = result          # 항상 PostgreSQL
        self.db_nm  = db_nm
        self._tf    = _flavor(target)
        self._excl  = {"oracle": ORACLE_EXCL, "postgres": PG_EXCL,
                       "mysql": MYSQL_EXCL, "mssql": MSSQL_EXCL}.get(self._tf, ORACLE_EXCL)
        self.schemas = []

    # ── 차수 채번 / 등록 ───────────────────────────────────────
    def _next_chasu(self) -> int:
        sql = (f'SELECT COALESCE(MAX("CHASU"), 0) + 1 '
               f'FROM ENC_DIC_CHASU_MAS" WHERE "DB_NM" = %s')
        rows = self.target.execute_query(sql, (self.db_nm,))
        return int(rows[0][0]) if rows else 1

    def _reg_chasu(self, chasu: int):
        sql = (f'INSERT INTO ENC_DIC_CHASU_MAS '
               f'("CHASU","DB_NM","CR_DT") VALUES (%s,%s,NOW())')
        self.target.execute_dml(sql, (chasu, self.db_nm))

    # ── 테이블 수집 ────────────────────────────────────────────
    def _collect_tables(self, chasu: int):
        if self._tf == "oracle":
            sel = f"""
                SELECT A.OWNER, A.TABLE_NAME, B.COMMENTS,
                       NVL(C.MBYTES, 0), D.CREATED,
                       CASE WHEN A.PARTITIONED = 'YES' THEN 'Y' ELSE 'N' END
                FROM ALL_TABLES A
                LEFT JOIN ALL_TAB_COMMENTS B
                    ON A.OWNER = B.OWNER AND A.TABLE_NAME = B.TABLE_NAME AND B.TABLE_TYPE = 'TABLE'
                LEFT JOIN (
                    SELECT OWNER, SEGMENT_NAME, TRUNC(SUM(BYTES)/1024/1024) MBYTES
                    FROM DBA_SEGMENTS
                    WHERE SEGMENT_TYPE IN ('TABLE','TABLE PARTITION')
                    GROUP BY OWNER, SEGMENT_NAME
                ) C ON A.OWNER = C.OWNER AND A.TABLE_NAME = C.SEGMENT_NAME
                LEFT JOIN (
                    SELECT OWNER, OBJECT_NAME, MIN(CREATED) CREATED
                    FROM DBA_OBJECTS
                    WHERE OBJECT_TYPE LIKE 'TABLE%'
                    GROUP BY OWNER, OBJECT_NAME
                ) D ON A.OWNER = D.OWNER AND A.TABLE_NAME = D.OBJECT_NAME
                WHERE A.OWNER NOT IN ({self._excl})
                ORDER BY A.OWNER, A.TABLE_NAME
            """
        else:
            sel = f"""
                SELECT t.table_schema, t.table_name,
                       obj_description(
                           (quote_ident(t.table_schema)||'.'||quote_ident(t.table_name))::regclass
                       ),
                       TRUNC(pg_total_relation_size(
                           (quote_ident(t.table_schema)||'.'||quote_ident(t.table_name))::regclass
                       ) / 1024.0 / 1024),
                       NULL, 'N'
                FROM information_schema.tables t
                WHERE t.table_type = 'BASE TABLE'
                  AND t.table_schema NOT IN ({self._excl})
                ORDER BY t.table_schema, t.table_name
            """
        rows = self.target.execute_query(sel)
        if not rows:
            return

        self.result.execute_dml(
            f'DELETE FROM "{self.rs}"."ENC_DIC_TAB_MAS" WHERE "DB_NM"=%s AND "CHASU"=%s',
            (self.db_nm, chasu)
        )
        ins = (f'INSERT INTO "{self.rs}"."ENC_DIC_TAB_MAS" '
               f'("CHASU","DB_NM","OWNER_NM","TAB_NM","TAB_COMMENT","TAB_SIZE","TAB_CREATED","PART_YN") '
               f'VALUES (%s,%s,%s,%s,%s,%s,%s,%s)')
        self.result.execute_many(ins,
            [(chasu, self.db_nm, r[0], r[1], r[2] or "", r[3], r[4], r[5]) for r in rows])
        print(f"[DIC] 테이블 {len(rows)}건 수집")

    # ── 컬럼 수집 ──────────────────────────────────────────────
    def _collect_columns(self, chasu: int):
        if self._tf == "oracle":
            sel = f"""
                SELECT A.OWNER, A.TABLE_NAME, A.COLUMN_NAME, B.COMMENTS,
                       A.DATA_TYPE, A.COLUMN_ID, A.NULLABLE,
                       A.DATA_LENGTH, A.DATA_PRECISION, A.DATA_SCALE
                FROM ALL_TAB_COLUMNS A
                LEFT JOIN ALL_COL_COMMENTS B
                    ON A.OWNER = B.OWNER AND A.TABLE_NAME = B.TABLE_NAME AND A.COLUMN_NAME = B.COLUMN_NAME
                WHERE A.OWNER NOT IN ({self._excl})
                  AND NOT EXISTS (
                      SELECT 'X' FROM ALL_VIEWS V
                      WHERE A.OWNER = V.OWNER AND A.TABLE_NAME = V.VIEW_NAME
                  )
                ORDER BY A.OWNER, A.TABLE_NAME, A.COLUMN_ID
            """
        else:
            sel = f"""
                SELECT c.table_schema, c.table_name, c.column_name, pd.description,
                       c.data_type, c.ordinal_position,
                       CASE WHEN c.is_nullable = 'YES' THEN 'Y' ELSE 'N' END,
                       COALESCE(c.character_maximum_length, c.numeric_precision, c.datetime_precision),
                       c.numeric_precision, c.numeric_scale
                FROM information_schema.columns c
                LEFT JOIN pg_stat_all_tables st
                    ON st.schemaname = c.table_schema AND st.relname = c.table_name
                LEFT JOIN pg_catalog.pg_description pd
                    ON pd.objoid = st.relid AND pd.objsubid = c.ordinal_position
                WHERE c.table_schema NOT IN ({self._excl})
                  AND NOT EXISTS (
                      SELECT 1 FROM information_schema.views v
                      WHERE v.table_schema = c.table_schema AND v.table_name = c.table_name
                  )
                ORDER BY c.table_schema, c.table_name, c.ordinal_position
            """
        rows = self.target.execute_query(sel)
        if not rows:
            return

        self.result.execute_dml(
            f'DELETE FROM "{self.rs}"."ENC_DIC_COL_MAS" WHERE "DB_NM"=%s AND "CHASU"=%s',
            (self.db_nm, chasu)
        )
        ins = (f'INSERT INTO "{self.rs}"."ENC_DIC_COL_MAS" '
               f'("CHASU","DB_NM","OWNER_NM","TAB_NM","COL_NM","COL_COMMENT",'
               f'"DATA_TYPE","COL_SEQ","NULL_YN","DATA_LEN","DATA_PRECISION","DATA_SCALE","CR_DT") '
               f'VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,NOW())')
        self.result.execute_many(ins,
            [(chasu, self.db_nm, r[0], r[1], r[2], r[3] or "",
              r[4], r[5], r[6], r[7], r[8], r[9]) for r in rows])
        print(f"[DIC] 컬럼 {len(rows)}건 수집")

    # ── 세그먼트 수집 ──────────────────────────────────────────
    def _collect_segments(self, chasu: int):
        if self._tf == "oracle":
            sel = f"""
                SELECT OWNER, SEGMENT_NAME, SEGMENT_TYPE, BYTES
                FROM DBA_SEGMENTS
                WHERE SEGMENT_TYPE IN ('TABLE','TABLE PARTITION','INDEX','INDEX PARTITION')
                  AND OWNER NOT IN ({self._excl})
            """
        else:
            sel = f"""
                SELECT n.nspname, c.relname,
                       CASE c.relkind
                           WHEN 'r' THEN 'TABLE'       WHEN 'p' THEN 'TABLE PARTITION'
                           WHEN 'i' THEN 'INDEX'        WHEN 'I' THEN 'INDEX PARTITION'
                       END,
                       pg_relation_size(c.oid)
                FROM pg_class c
                JOIN pg_namespace n ON n.oid = c.relnamespace
                WHERE c.relkind IN ('r','p','i','I')
                  AND n.nspname NOT IN ({self._excl})
            """
        rows = self.target.execute_query(sel)
        if not rows:
            return

        self.result.execute_dml(
            f'DELETE FROM "{self.rs}"."ENC_DIC_SEGMENTS" WHERE "DB_NM"=%s AND "CHASU"=%s',
            (self.db_nm, chasu)
        )
        ins = (f'INSERT INTO "{self.rs}"."ENC_DIC_SEGMENTS" '
               f'("CHASU","DB_NM","OWNER_NM","SEGMENT_NM","SEGMENT_TYPE","SEGMENT_SIZE") '
               f'VALUES (%s,%s,%s,%s,%s,%s)')
        self.result.execute_many(ins,
            [(chasu, self.db_nm, r[0], r[1], r[2], r[3]) for r in rows])
        print(f"[DIC] 세그먼트 {len(rows)}건 수집")

    # ── 제약조건 수집 ──────────────────────────────────────────
    def _collect_constraints(self, chasu: int):
        if self._tf == "oracle":
            cons_sel = f"""
                SELECT OWNER, TABLE_NAME, CONSTRAINT_NAME, CONSTRAINT_TYPE
                FROM ALL_CONSTRAINTS
                WHERE CONSTRAINT_TYPE IN ('P','U','R') AND OWNER NOT IN ({self._excl})
            """
            col_sel = f"""
                SELECT A.OWNER, A.CONSTRAINT_NAME, A.COLUMN_NAME, A.POSITION
                FROM ALL_CONS_COLUMNS A
                JOIN ALL_CONSTRAINTS B ON A.OWNER = B.OWNER AND A.CONSTRAINT_NAME = B.CONSTRAINT_NAME
                WHERE B.CONSTRAINT_TYPE IN ('P','U','R') AND A.OWNER NOT IN ({self._excl})
            """
        else:
            cons_sel = f"""
                SELECT tc.table_schema, tc.table_name, tc.constraint_name,
                       CASE tc.constraint_type
                           WHEN 'PRIMARY KEY' THEN 'P' WHEN 'UNIQUE' THEN 'U' WHEN 'FOREIGN KEY' THEN 'R'
                       END
                FROM information_schema.table_constraints tc
                WHERE tc.constraint_type IN ('PRIMARY KEY','UNIQUE','FOREIGN KEY')
                  AND tc.table_schema NOT IN ({self._excl})
            """
            col_sel = f"""
                SELECT kcu.table_schema, kcu.constraint_name, kcu.column_name, kcu.ordinal_position
                FROM information_schema.key_column_usage kcu
                JOIN information_schema.table_constraints tc
                    ON kcu.constraint_name = tc.constraint_name AND kcu.table_schema = tc.table_schema
                WHERE tc.constraint_type IN ('PRIMARY KEY','UNIQUE','FOREIGN KEY')
                  AND kcu.table_schema NOT IN ({self._excl})
            """

        rows = self.target.execute_query(cons_sel)
        if rows:
            self.result.execute_dml(
                f'DELETE FROM "{self.rs}"."ENC_DIC_CONS" WHERE "DB_NM"=%s AND "CHASU"=%s',
                (self.db_nm, chasu)
            )
            ins = (f'INSERT INTO "{self.rs}"."ENC_DIC_CONS" '
                   f'("CHASU","DB_NM","OWNER_NM","TABLE_NM","CONS_NM","CONS_TYPE") '
                   f'VALUES (%s,%s,%s,%s,%s,%s)')
            self.result.execute_many(ins,
                [(chasu, self.db_nm, r[0], r[1], r[2], r[3]) for r in rows])

        rows2 = self.target.execute_query(col_sel)
        if rows2:
            self.result.execute_dml(
                f'DELETE FROM "{self.rs}"."ENC_DIC_CONS_COL" WHERE "DB_NM"=%s AND "CHASU"=%s',
                (self.db_nm, chasu)
            )
            ins2 = (f'INSERT INTO "{self.rs}"."ENC_DIC_CONS_COL" '
                    f'("CHASU","DB_NM","OWNER_NM","CONS_NM","COL_NM","POS") '
                    f'VALUES (%s,%s,%s,%s,%s,%s)')
            self.result.execute_many(ins2,
                [(chasu, self.db_nm, r[0], r[1], r[2], r[3]) for r in rows2])

        print(f"[DIC] 제약조건 {len(rows or [])}건 / 제약컬럼 {len(rows2 or [])}건 수집")

    # ── 인덱스 수집 ────────────────────────────────────────────
    def _collect_indexes(self, chasu: int):
        if self._tf == "oracle":
            sel = f"""
                SELECT OWNER, INDEX_NAME, TABLE_OWNER, TABLE_NAME
                FROM ALL_INDEXES
                WHERE OWNER NOT IN ({self._excl})
            """
        else:
            sel = f"""
                SELECT schemaname, indexname, schemaname, tablename
                FROM pg_indexes
                WHERE schemaname NOT IN ({self._excl})
            """
        rows = self.target.execute_query(sel)
        if not rows:
            return

        self.result.execute_dml(
            f'DELETE FROM "{self.rs}"."ENC_DIC_IND_MAS" WHERE "DB_NM"=%s AND "CHASU"=%s',
            (self.db_nm, chasu)
        )
        ins = (f'INSERT INTO "{self.rs}"."ENC_DIC_IND_MAS" '
               f'("CHASU","DB_NM","OWNER_NM","INDEX_NM","TAB_OWNER_NM","TAB_NM") '
               f'VALUES (%s,%s,%s,%s,%s,%s)')
        self.result.execute_many(ins,
            [(chasu, self.db_nm, r[0], r[1], r[2], r[3]) for r in rows])
        print(f"[DIC] 인덱스 {len(rows)}건 수집")

    # ── 전체 실행 ──────────────────────────────────────────────
    def collect(self, schema: str = "public") -> int:
        """딕셔너리 전체 수집. 완료된 CHASU 반환."""
        self.rs = schema
        chasu = self._next_chasu()
        self._reg_chasu(chasu)
        print(f"[Step1] 시작 — db_nm={self.db_nm}  schema={schema}  chasu={chasu}  target={self._tf}")
        self._collect_tables(chasu)
        self._collect_columns(chasu)
        self._collect_segments(chasu)
        self._collect_constraints(chasu)
        self._collect_indexes(chasu)
        print(f"[Step1] 완료 — chasu={chasu}")
        return chasu


# ── 실행 예시 ──────────────────────────────────────────────────
# if __name__ == "__main__":
#     factory = DBConFactory()
#     factory.register_oracle(
#         "src", host="172.16.105.117", port=1521,
#         service_name="PDB1", user="E2218030", password="password"
#     )
#     factory.register_postgres(
#         "res", host="172.16.105.117", port=5432,
#         database="postgres", user="e2218030", password="p2218030"
#     )

#     dc = DicCollector(
#         target = factory.get("src"),
#         result = factory.get("res"),
#         db_nm  = "01",
#         # schema = "public",
#     )
#     dc.collect()
#     factory.close_all()
